# Emotion Classification with Attention
**Dataset:** GoEmotions (Google) mapped to 6 emotions: joy, sadness, anger, fear, surprise, disgust
**Models:** LSTM, GRU, BiLSTM + Attention (GloVe embeddings), and fine-tuned DistilBERT (HuggingFace Trainer)
**Deployment:** Gradio app with confidence chart and supportive messages

> Runtime: use a GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!pip -q install datasets transformers accelerate gensim gradio scikit-learn seaborn

In [ ]:
import re, random, collections, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
C2I = {c: i for i, c in enumerate(CLASSES)}
MAX_LEN, EMB_DIM, BATCH = 40, 100, 64
print("Device:", DEVICE)

## 1. Load and explore GoEmotions

In [ ]:
from datasets import load_dataset
raw = load_dataset("google-research-datasets/go_emotions", "simplified")
names = raw["train"].features["labels"].feature.names
print(raw)
print("Original labels:", len(names), names)
raw["train"].to_pandas().head()

## 2. Map the 27 emotions to the 6 target classes (Ekman grouping)
Neutral and examples that map to more than one target class are dropped.

In [ ]:
EKMAN = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love",
            "optimism", "pride", "admiration", "desire", "relief", "caring"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral": ["neutral"],
}
FINE2EKMAN = {fine: ek for ek, fines in EKMAN.items() for fine in fines}
assert set(FINE2EKMAN) == set(names), set(names) ^ set(FINE2EKMAN)

def to_df(split):
    rows = []
    for text, labs in zip(split["text"], split["labels"]):
        mapped = {FINE2EKMAN[names[i]] for i in labs} - {"neutral"}
        if len(mapped) == 1:
            rows.append((text, mapped.pop()))
    df = pd.DataFrame(rows, columns=["text", "emotion"])
    df["label"] = df["emotion"].map(C2I)
    return df

train_df, val_df, test_df = to_df(raw["train"]), to_df(raw["validation"]), to_df(raw["test"])
print(len(train_df), len(val_df), len(test_df))
train_df.sample(5, random_state=1)

In [ ]:
counts = train_df["emotion"].value_counts().reindex(CLASSES)
print(counts)
plt.figure(figsize=(7, 3.5))
sns.barplot(x=counts.index, y=counts.values, palette="viridis")
plt.title("Training set class distribution"); plt.ylabel("Samples"); plt.show()

train_df["n_words"] = train_df["text"].str.split().str.len()
print(train_df["n_words"].describe())

## 3. Clean, tokenize, build vocabulary, pad to fixed length

In [ ]:
def clean(t):
    t = t.lower()
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"\[(name|religion)\]", " ", t)
    return t

def tokenize(t):
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[!?]", clean(t))

counter = collections.Counter(w for t in train_df["text"] for w in tokenize(t))
vocab = {"<pad>": 0, "<unk>": 1}
for w, c in counter.items():
    if c >= 2:
        vocab[w] = len(vocab)
print("Vocab size:", len(vocab))

def encode(texts):
    X = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, 1) for w in tokenize(t)][:MAX_LEN]
        if not ids: ids = [1]
        X[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return torch.tensor(X), torch.tensor(lengths)

def make_loader(df, shuffle):
    X, L = encode(df["text"].tolist())
    ds = TensorDataset(X, L, torch.tensor(df["label"].values))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

train_loader, val_loader, test_loader = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)

## 4. Embedding layer initialized with pre-trained GloVe

In [ ]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-100")   # 100-d GloVe vectors

emb_matrix = np.random.normal(0, 0.1, (len(vocab), EMB_DIM)).astype("float32")
emb_matrix[0] = 0
hits = 0
for w, i in vocab.items():
    if w in glove:
        emb_matrix[i] = glove[w]; hits += 1
print(f"GloVe coverage: {hits}/{len(vocab)} ({hits/len(vocab):.1%})")

## 5. Models: LSTM, GRU, BiLSTM + Attention

In [ ]:
class SeqClassifier(nn.Module):
    def __init__(self, emb, kind="lstm", hidden=128, bidir=False, attention=False, n_classes=6, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb), freeze=False, padding_idx=0)
        rnn_cls = nn.LSTM if kind == "lstm" else nn.GRU
        self.rnn = rnn_cls(emb.shape[1], hidden, batch_first=True, bidirectional=bidir)
        out_dim = hidden * (2 if bidir else 1)
        self.attention = attention
        if attention:
            self.att_proj = nn.Linear(out_dim, out_dim)
            self.att_v = nn.Linear(out_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(out_dim, n_classes)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        weights = None
        if self.attention:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
            scores = self.att_v(torch.tanh(self.att_proj(out))).squeeze(-1)
            scores = scores.masked_fill(x == 0, -1e9)          # ignore padding
            weights = torch.softmax(scores, dim=1)             # (B, T)
            feat = (out * weights.unsqueeze(-1)).sum(1)        # weighted sum of hidden states
        else:
            if isinstance(h, tuple): h = h[0]                  # LSTM returns (h, c)
            feat = torch.cat([h[-2], h[-1]], 1) if self.rnn.bidirectional else h[-1]
        return self.fc(self.drop(feat)), weights

# class weights to handle imbalance (disgust is rare)
cnt = np.bincount(train_df["label"], minlength=6)
class_w = torch.tensor(cnt.sum() / (6 * cnt), dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_w)

def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, l, y in loader:
            logits, _ = model(x.to(DEVICE), l)
            ps += logits.argmax(1).cpu().tolist(); ys += y.tolist()
    return np.array(ys), np.array(ps)

def train_model(model, epochs=10, lr=2e-3, name="model"):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state = 0, None
    for ep in range(1, epochs + 1):
        model.train(); total = 0
        for x, l, y in train_loader:
            opt.zero_grad()
            logits, _ = model(x.to(DEVICE), l)
            loss = criterion(logits, y.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total += loss.item()
        yv, pv = predict(model, val_loader)
        f1 = f1_score(yv, pv, average="macro")
        print(f"[{name}] epoch {ep:2d} | loss {total/len(train_loader):.4f} | val acc {accuracy_score(yv, pv):.4f} | val macro-F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

In [ ]:
lstm_model = train_model(SeqClassifier(emb_matrix, "lstm"), name="LSTM")

In [ ]:
gru_model = train_model(SeqClassifier(emb_matrix, "gru"), name="GRU")

In [ ]:
att_model = train_model(SeqClassifier(emb_matrix, "lstm", bidir=True, attention=True), name="BiLSTM+Attn")

## 6. Evaluation: accuracy, macro F1, per-class precision/recall, confusion matrix

In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    results[name] = {"Accuracy": acc, "Macro F1": f1}
    print(f"===== {name} =====\nAccuracy: {acc:.4f} | Macro F1: {f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f"{name} - confusion matrix (row-normalized)")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

for name, m in [("LSTM", lstm_model), ("GRU", gru_model), ("BiLSTM + Attention", att_model)]:
    evaluate(name, *predict(m, test_loader))

## 7. Attention heatmaps: which words drive each prediction?

In [ ]:
def explain(text, model=att_model):
    model.eval()
    toks = tokenize(text)[:MAX_LEN] or ["<unk>"]
    ids = torch.tensor([[vocab.get(w, 1) for w in toks]])
    with torch.no_grad():
        logits, w = model(ids.to(DEVICE), torch.tensor([len(toks)]))
    probs = torch.softmax(logits, 1)[0].cpu()
    pred = CLASSES[probs.argmax().item()]
    plt.figure(figsize=(max(6, len(toks) * 0.8), 1.8))
    sns.heatmap(w[0].cpu().numpy()[None, :len(toks)], annot=True, fmt=".2f", cmap="Reds",
                cbar=False, xticklabels=toks, yticklabels=[f"{pred} ({probs.max():.0%})"])
    plt.title("Attention weights per token"); plt.yticks(rotation=0); plt.show()

for s in ["I am so happy and grateful for this amazing news!",
          "I miss him so much and I feel completely hopeless",
          "This is disgusting, I can't believe you did that",
          "I'm nervous and scared about the exam tomorrow",
          "I can't believe this just happened, what a shock!",
          "You lied to me and I am furious"]:
    explain(s)

## 8. Fine-tune DistilBERT with the HuggingFace Trainer

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

CKPT = "distilbert-base-uncased"
hf_tok = AutoTokenizer.from_pretrained(CKPT)

def to_hf(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    return ds.map(lambda b: hf_tok(b["text"], truncation=True, max_length=64), batched=True)

hf_train, hf_val, hf_test = to_hf(train_df), to_hf(val_df), to_hf(test_df)

hf_model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=6,
    id2label={i: c for i, c in enumerate(CLASSES)}, label2id=C2I)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_w.to(outputs.logits.device))(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="distilbert_emotion", num_train_epochs=3, learning_rate=3e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="macro_f1", fp16=torch.cuda.is_available(),
    report_to="none", seed=SEED)

trainer = WeightedTrainer(model=hf_model, args=args, train_dataset=hf_train, eval_dataset=hf_val,
                          processing_class=hf_tok, data_collator=DataCollatorWithPadding(hf_tok),
                          compute_metrics=compute_metrics)
trainer.train()

In [ ]:
out = trainer.predict(hf_test)
evaluate("DistilBERT (fine-tuned)", out.label_ids, out.predictions.argmax(-1))

## 9. Compare all four models

In [ ]:
comp = pd.DataFrame(results).T.round(4).sort_values("Macro F1", ascending=False)
display(comp)
comp.plot(kind="bar", figsize=(8, 4), ylim=(0, 1)); plt.title("Model comparison (test set)")
plt.xticks(rotation=20); plt.show()

## 10. Deployment: Gradio app with supportive messages
The app returns the dominant emotion, a confidence chart across all 6 classes, and a message based on the emotion:
- **Sadness:** comfort + a small tip
- **Joy:** amplifies the happiness
- **Fear/nervous:** interactive breathing/grounding exercise and a reflection question

In [ ]:
trainer.save_model("emotion_distilbert")
hf_tok.save_pretrained("emotion_distilbert")

In [ ]:
%%writefile app.py
import os
import random

import gradio as gr
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = os.environ.get("MODEL_DIR", "emotion_distilbert")  # local folder or HF Hub id
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
LABELS = [model.config.id2label[i] for i in range(model.config.num_labels)]

SUPPORT = {
    "sadness": {
        "emoji": "💙",
        "messages": [
            "I'm sorry you're feeling low. It's okay to feel this way, you don't have to push it away. Be gentle with yourself today. Tomorrow is a fresh start, and it can be better.",
            "That sounds heavy, and your feelings are valid. You've gotten through hard days before, and this one will pass too. Tomorrow will be better.",
            "Sadness shows that something mattered to you. Take your time. You're not alone in this, and brighter days are coming.",
        ],
        "tips": [
            "Write down one small thing that went okay today, even a tiny one.",
            "Reach out to someone you trust, even just to say hi.",
            "Drink some water, step outside for a few minutes, or play a song you love.",
            "Rest is productive too. Go easy on your to-do list today.",
        ],
    },
    "joy": {
        "emoji": "🎉",
        "messages": [
            "That's wonderful to hear! You deserve this happiness. Soak it all in!",
            "Love this energy! Moments like this are worth celebrating.",
            "So glad you're feeling great! Let this joy carry you through the whole day and beyond.",
        ],
        "tips": [
            "Share the good news with someone. Joy grows when it's shared!",
            "Write down what made you happy so you can revisit it on tougher days.",
            "Treat yourself to something small to celebrate the moment.",
            "Pay it forward: send a kind message to someone today.",
        ],
    },
    "fear": {
        "emoji": "🌿",
        "messages": [
            "Feeling nervous is normal, and it usually means something matters to you. Let's slow things down together for a moment.",
            "It's okay to feel anxious. You don't have to figure everything out right now. Just the next small step.",
        ],
        "tips": [
            "Breathe with me: in for 4 seconds, hold for 4, out for 6. Repeat 3 times.",
            "Grounding (5-4-3-2-1): name 5 things you see, 4 you can touch, 3 you hear, 2 you smell, 1 you taste.",
            "Ask yourself: what is ONE thing I can control right now? Start there.",
            "Ask yourself: what would I tell a friend who felt this way?",
        ],
    },
    "anger": {
        "emoji": "🔥",
        "messages": [
            "It sounds like something really got to you. Your frustration is valid.",
            "Anger often points to something that crossed a line for you. It's okay to feel it. Let's cool down before acting on it.",
        ],
        "tips": [
            "Pause before replying or reacting. Give it 10 slow breaths.",
            "Take a short walk or splash cold water on your face.",
            "Write out everything you feel, then decide what is actually worth acting on.",
        ],
    },
    "surprise": {
        "emoji": "✨",
        "messages": [
            "Wow, that sounds unexpected! Take a moment to let it sink in.",
            "Life certainly keeps things interesting! However you feel about it, you can handle it.",
        ],
        "tips": [
            "Give yourself a minute to process before making any decisions.",
            "Tell someone about it. Talking helps make sense of surprises.",
        ],
    },
    "disgust": {
        "emoji": "🍃",
        "messages": [
            "That sounds really unpleasant. It's fair to want distance from whatever caused it.",
            "Some things just rub us the wrong way, and that's valid. Let's reset your mood.",
        ],
        "tips": [
            "Step away from whatever caused it and change your environment for a bit.",
            "Do something refreshing: fresh air, a shower, or a favorite snack.",
        ],
    },
}

LOW_CONFIDENCE = "I'm not totally sure how you're feeling, and that's okay. Feel free to tell me a bit more. 🙂"


def build_message(emotion, confidence):
    if emotion is None:
        return ""
    if confidence < 0.40:
        return LOW_CONFIDENCE
    s = SUPPORT[emotion]
    return f"### {s['emoji']} {random.choice(s['messages'])}\n\n**💡 Try this:** {random.choice(s['tips'])}"


def classify(text):
    if not text or not text.strip():
        return {}, "Type how you're feeling and I'll tell you what I sense.", None, 0.0
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0].tolist()
    scores = {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}
    top = max(scores, key=scores.get)
    return scores, build_message(top, scores[top]), top, scores[top]


def another(emotion, confidence):
    return build_message(emotion, confidence or 0.0)


with gr.Blocks(title="Emotion Classifier") as demo:
    gr.Markdown("# 💬 Emotion Classification with Attention\nWrite how you feel in your own words.")
    emotion_state = gr.State(None)
    conf_state = gr.State(0.0)
    inp = gr.Textbox(lines=3, label="Your text", placeholder="e.g. I have a big interview tomorrow and I can't stop worrying...")
    with gr.Row():
        btn = gr.Button("Analyze", variant="primary")
        again = gr.Button("Give me another message")
    with gr.Row():
        chart = gr.Label(num_top_classes=6, label="Confidence across emotions")
        msg = gr.Markdown()
    outputs = [chart, msg, emotion_state, conf_state]
    btn.click(classify, inp, outputs)
    inp.submit(classify, inp, outputs)
    again.click(another, [emotion_state, conf_state], msg)

if __name__ == "__main__":
    demo.launch()

In [ ]:
# Launch inside Colab (gives a public link)
import importlib, app
importlib.reload(app)
app.demo.launch(share=True, debug=False)

In [ ]:
# Optional: download the model + app to deploy on Hugging Face Spaces
import shutil
shutil.make_archive("emotion_distilbert", "zip", "emotion_distilbert")
from google.colab import files
files.download("emotion_distilbert.zip"); files.download("app.py")

### Deploy on Hugging Face Spaces (free)
1. Create a new Space (SDK: Gradio) and upload `app.py` plus the unzipped `emotion_distilbert/` folder.
2. Add a `requirements.txt` with: `torch`, `transformers`, `gradio`.